In [50]:
import time
import mujoco
import mujoco.viewer
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv,VecNormalize
from env_for_quadruped import QPEnv


In [51]:
MODEL_PATH="check_points/qp_ppo_1000000_steps"
VECNORM_PATH= "check_points/qp_ppo_vecnormalize_1000000_steps.pkl"
XML_PATH="quadruped.xml"


In [52]:
def main():
    raw_env=QPEnv(mjcf_path=XML_PATH)
    model=PPO.load(MODEL_PATH)
    vec_env=DummyVecEnv([lambda: raw_env])
    vec_env=VecNormalize.load(VECNORM_PATH,vec_env)
    vec_env.training=False
    vec_env.norm_reward=False
    obs=vec_env.reset()

    with mujoco.viewer.launch_passive(raw_env.model,raw_env.data) as viewer:
        print("Viewr is open - close the window to stop. ")
        while viewer.is_running():
            loop_start = time.time()
            action,_=model.predict(obs,deterministic=True)
            obs,reward,done,info=vec_env.step(action)
            viewer.sync()
            if done[0]:
                obs=vec_env.reset()
            elapsed=time.time()-loop_start
            if elapsed <raw_env.model.opt.timestep:
                time.sleep(raw_env.model.opt.timestep-elapsed)

if __name__=="__main__":
    main()

Viewr is open - close the window to stop. 


In [53]:
from env_for_quadruped import QPEnv
env = QPEnv(mjcf_path="quadruped.xml")
obs, _ = env.reset()
print(obs.shape)  # should print (5,)

(27,)
